# Resumo dos dados (mu = 0.07)

Retrato do arquivo diário antes do GLM: quantas vacas e casos, como os casos se distribuem
por baia e no tempo, quanto dura uma infecção e quanta exposição as vacas receberam na semana
antes de cada avaliação.

Lê o arquivo gerado por `python -m src.daily 0.07`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.load import PROCESSED_DIR

MU = "0.07"
daily = (
    pd.read_parquet(PROCESSED_DIR / f"daily_mu{MU}.parquet")
    .sort_values(["Agent", "date"])
    .reset_index(drop=True)
)

In [ ]:
# cores e estilo dos gráficos
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "font.family": ["Segoe UI", "DejaVu Sans"], "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": "#c3c2b7", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True,
    "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlecolor": INK,
})

## Visão geral

As patas são avaliadas uma vez por semana, às quintas, e o status vale até a avaliação seguinte.
Por isso todo caso novo cai numa quinta.

In [ ]:
assessments = daily[daily["date"].dt.dayofweek == 3]
assert daily.loc[daily["C"] == 1, "date"].dt.dayofweek.eq(3).all()

pd.Series({
    "vacas": daily["Agent"].nunique(),
    "vaca-dias": len(daily),
    "primeiro dia": daily["date"].min().date(),
    "último dia": daily["date"].max().date(),
    "avaliações (quintas)": assessments["date"].nunique(),
    "casos novos (vaca)": daily["C"].sum(),
    "vacas com caso novo": daily.loc[daily["C"] == 1, "Agent"].nunique(),
    "casos novos (perna)": daily["Ct"].sum(),
    "já infectadas no primeiro dia": (daily["first_day"] & daily["INFECTED"].eq(1)).sum(),
    "trocas de baia": daily["barn_move"].sum(),
    "vaca-dias sem exposição": daily["Exp"].isna().sum(),
}, name=f"mu = {MU}").to_frame()

## Por baia

A soma das vacas passa do total porque algumas vacas mudam de baia.

In [ ]:
daily.groupby("barn").agg(
    vacas=("Agent", "nunique"),
    vaca_dias=("date", "size"),
    casos_vaca=("C", "sum"),
    casos_perna=("Ct", "sum"),
)

## Ao longo do tempo

Casos novos e proporção de vacas infectadas em cada avaliação.

In [ ]:
weekly = assessments.groupby("date").agg(
    vacas=("Agent", "nunique"), infectadas=("INFECTED", "sum"), casos_novos=("C", "sum"),
)
weekly["infectadas_pct"] = 100 * weekly["infectadas"] / weekly["vacas"]

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
top.bar(weekly.index, weekly["casos_novos"], width=4, color=BLUE)
top.set_title("Casos novos por avaliação")
top.yaxis.set_major_locator(MaxNLocator(integer=True))

bottom.plot(weekly.index, weekly["infectadas_pct"], color=BLUE, lw=1.5, marker="o", ms=4)
bottom.set_title("Vacas infectadas no dia da avaliação (%)")
bottom.set_ylim(bottom=0)
bottom.xaxis.set_major_locator(mdates.MonthLocator())
bottom.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))
fig.tight_layout()
weekly

## Duração das infecções

Um episódio é uma sequência de dias seguidos com a vaca infectada. Episódios que vão até o último
dia da vaca não têm fim conhecido e ficam fora do gráfico, assim como os das vacas que já chegaram
infectadas. A estimativa da taxa de recuperação, que trata esses casos, fica para o passo 3.

In [ ]:
infected = daily["INFECTED"].eq(1)
new_episode = infected.ne(infected.groupby(daily["Agent"]).shift())
daily["episode"] = new_episode.cumsum()
daily["last_day"] = daily["date"].eq(daily.groupby("Agent")["date"].transform("max"))

episodes = daily[infected].groupby("episode").agg(
    Agent=("Agent", "first"),
    start=("date", "min"),
    end=("date", "max"),
    starts_with_case=("C", "first"),
    starts_on_first_day=("first_day", "first"),
    ongoing=("last_day", "max"),
)
episodes["days"] = (episodes["end"] - episodes["start"]).dt.days + 1
episodes["weeks"] = np.ceil(episodes["days"] / 7).astype(int)

pd.crosstab(
    episodes["starts_with_case"].map({1: "começa com caso novo", 0: "já infectada no 1º dia"}),
    episodes["ongoing"].map({True: "sem fim conhecido", False: "terminou"}),
    margins=True, margins_name="total",
)

In [ ]:
done = episodes[episodes["starts_with_case"].eq(1) & ~episodes["ongoing"]]
counts = done["weeks"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(counts.index, counts.to_numpy(), width=0.6, color=BLUE)
ax.set_title(f"Duração das infecções que começaram e terminaram nos dados (n = {len(done)})")
ax.set_xlabel("semanas infectada")
ax.set_xticks(range(1, counts.index.max() + 1))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
fig.tight_layout()
done["weeks"].describe().round(1)

## Exposição na semana antes da avaliação

Como a infecção aconteceu em algum dia da semana anterior à avaliação, a exposição de cada vaca é
somada nos 7 dias antes da quinta (o dia da avaliação fica de fora). Entram só as semanas em que a
vaca estava saudável na véspera, e só janelas com os 7 dias de exposição.

Esta é uma comparação crua, sem modelo. Qual janela usar no GLM ainda está para decidir no passo 4.

In [ ]:
by_cow = daily.set_index("date").groupby("Agent")["Exp"]
daily["exp_week"] = by_cow.rolling("7D", closed="left", min_periods=1).sum().to_numpy()
daily["exp_week_days"] = by_cow.rolling("7D", closed="left").count().to_numpy()

at_risk_weeks = daily[daily["date"].dt.dayofweek.eq(3) & daily["at_risk"]]
weeks = at_risk_weeks[at_risk_weeks["exp_week_days"].eq(7)]
print(f"{len(at_risk_weeks)} semanas em risco; {len(weeks)} com os 7 dias de exposição")

groups = {0: "sem caso novo", 1: "caso novo"}
weeks.groupby("C")["exp_week"].agg(
    semanas="size",
    exposicao_zero=lambda s: s.eq(0).sum(),
    zero_pct=lambda s: round(100 * s.eq(0).mean(), 1),
    mediana_se_positiva=lambda s: s[s > 0].median(),
).rename(index=groups)

In [ ]:
LOW = 1e-2
fig, ax = plt.subplots(figsize=(9, 3))
rng = np.random.default_rng(0)
for row, (c, color) in enumerate([(0, BLUE), (1, ORANGE)]):
    exp = weeks.loc[weeks["C"].eq(c), "exp_week"]
    pos = exp[exp >= LOW]
    big = c == 1
    ax.scatter(pos, row + rng.uniform(-0.18, 0.18, len(pos)), s=30 if big else 10,
               color=color, alpha=0.9 if big else 0.3, edgecolor="none")
    ax.plot([pos.median()] * 2, [row - 0.3, row + 0.3], color=INK, lw=1.5)
    zero = exp.eq(0).sum()
    ax.annotate(f"zero: {zero} de {len(exp)} ({100 * zero / len(exp):.0f}%)",
                (1.01, row), xycoords=("axes fraction", "data"), va="center", color=INK_2)

ax.set_xscale("log")
ax.set_xlim(left=LOW)
ax.set_yticks([0, 1], list(groups.values()))
ax.set_ylim(-0.6, 1.6)
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
ax.set_title("Exposição somada nos 7 dias antes da avaliação (escala log; traço = mediana)")
fig.tight_layout()
hidden = weeks["exp_week"].between(0, LOW, inclusive="neither").sum()
print(f"{hidden} semanas com exposição entre 0 e {LOW} ficam fora do gráfico")